# darts_forecast — the input data: what the model sees, when it is known, and how the data is split

This notebook explores the INPUTS of the Darts forecaster (`strategy.py`) before any model is fitted. Every
`BAR_MINUTES` (default 5) of the regular session, each of the five names gets one row of inputs:

| group | inputs | known |
|---|---|---|
| **intraday** | the grid bar's return, realized volatility, range, close location, signed-volume flow, volume against its time-of-day norm, return since the open, distance to the session VWAP | at the grid bar's close |
| **market** | the other names' average return | at the grid bar's close |
| **daily** | previous-day, 5-day and 20-day return (in volatility units), 20-day volatility, position in the 20-day range, the previous close location and relative volume, today's gap | at the previous close (the gap: at the open of today's first bar) |
| **options** | ATM implied vol, its daily change, skew, term structure (DoltHub chain); IV − HV (DoltHub vol history); customer put/call and volume surge (OCC); put/call open interest and volume, open-interest change, dealer gamma (MarketData.app) | at the end of the last stored day before the session, at most 5 days old |
| **flags** | one "has data" flag per option SOURCE: `has_dolt_iv`, `has_dolt_hv`, `has_occ`, `has_oi`; the daily inputs have none (§3) | with their source |
| **future** | time of day, weekday, the monthly-expiry week | in advance |
| **static** | which name (one global model learns all five) | always |

The model predicts the **target**: the return of the next `HORIZON` grid bars in volatility units (a z-score).

**The data split** lives in one place, `research.WINDOWS` (printed under the setup cell, drawn in §2). This notebook
reads **TUNE only**:
* prices of the TEST and HOLDOUT months are never loaded;
* their option days are read by `research.load` only to be dropped, since they belong to no TUNE session (check 8c
  confirms that none leaks into a TUNE value);
* §2 reads only their dates.

| § | what |
|---|---|
| 0 | the knobs and the split |
| 1 | load the TUNE panel and build the inputs |
| 2 | **the split**: which data trains, validates and tests |
| 3 | the input inventory: source, resolution, when each input is known, its lags, its flag, missing data |
| 4 | the target: z-scored grid returns and the volatility scale |
| 5 | a sample day: the 1-minute price, the grid, the intraday inputs, the daily steps |
| 6 | the daily and option inputs over time |
| 7 | how the inputs correlate |
| 8 | **timing and leakage checks** |
| 9 | a first univariate look (TUNE only; not a model) |

Nothing is fitted here; a run takes about a minute. Change a knob in §0 (for example `BAR_MINUTES = 15` or
`OPTIONS = "iv"`) and re-run it to see the inputs that setting gives the model.

In [ ]:
# If this import fails: select the "trading-simple" Jupyter kernel
# (see README "Quick start"), or run  uv sync --all-extras
import importlib.util
import warnings
from dataclasses import replace

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sqlalchemy import text

from trading_simple import plots, strategies
from trading_simple.config import load_settings
from trading_simple.db import make_engine

settings = load_settings()
engine = make_engine(settings)
# research.py holds the I/O, the tables and the evaluation the darts_forecast notebooks share,
# charts.py their figure helpers and checks.py the leakage checks; strategy.py (signals) never
# imports them.
_spec = importlib.util.spec_from_file_location(
    "darts_research", strategies.STRATEGIES_DIR / "darts_forecast" / "research.py"
)
rs = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(rs)
_spec = importlib.util.spec_from_file_location(
    "darts_charts", strategies.STRATEGIES_DIR / "darts_forecast" / "charts.py"
)
charts = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(charts)
_spec = importlib.util.spec_from_file_location(
    "darts_checks", strategies.STRATEGIES_DIR / "darts_forecast" / "checks.py"
)
ck = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(ck)
strat, SM = rs.STRAT, rs.SM  # the loaded strategy and strategy.py's own functions
P = strat.params
of = rs.options_module()  # burst_scan/options_free.py: the loaders of the free options store
# the option SOURCES of strategy.OPTION_SOURCES: each has its own has_<source> flag
STORE_OF = {  # source -> its folder in the options store
    "dolt_iv": "dolthub_chain",
    "dolt_hv": "dolthub_vol",
    "occ": "occ_volume",
    "oi": "marketdata_chain",
}
SOURCE_LABEL = {
    "dolt_iv": "DoltHub chain",
    "dolt_hv": "DoltHub vol history",
    "occ": "OCC volume",
    "oi": "MarketData chain",
}
SOURCE_OF = {"opt_" + n: src for src, names in SM["OPTION_SOURCES"].items() for n in names}
POS_C, NEG_C = plots.DIVERGING[-1][1], plots.DIVERGING[0][1]  # signed values: blue up, red down
GROUP_COLOR = {
    "intraday": plots.SERIES[0],
    "market": plots.SERIES[1],
    "daily": plots.SERIES[2],
    "options": plots.SERIES[4],
    "flag": plots.SERIES[6],
    "future": plots.SERIES[7],
}

# --- 0 · the knobs (defaults = config.yaml) -------------------------------------------------------
BAR_MINUTES = P["bar_minutes"]  # the forecast grid: one row of inputs per 5-minute bar
HORIZON = P["horizon"]  # grid bars the target covers (1 = the next bar)
OPTIONS = P["options"]  # "none" | "iv" | "all" option inputs
USE_DAILY = P["use_daily"]  # the daily inputs
USE_MARKET = P["use_market"]  # the other names' average return
VOL_DAYS = P["vol_days"]  # sessions behind the volatility scale
# the walk-forward, drawn by the split timeline of §2 (nothing is fitted in this notebook)
MODEL = P["model"]  # "gbm" | "gbm_clf" | "tide" | "tft": only tide / tft use a VALIDATION slice
TRAIN_DAYS = P["train_days"]  # sessions in each rolling training window
VAL_DAYS = P["val_days"]  # tide / tft: last sessions of each window, held out (early stopping)
RETRAIN_MINUTES = P["retrain_minutes"]  # 390 = before every session, 1950 = weekly, 0 = once
SAMPLE_NAME = "MU"  # the name §5 draws
SAMPLE_DAY = "2025-12-18"  # a TUNE session: MU's post-earnings gap; the IV drop lands a day later
CHECK_SEED = 0  # the random bars of checks 8b and 8f
NULL_SEED = 0  # the coin-flip market of §9
WARMUP = max(21, VOL_DAYS + 1)  # sessions that only build the daily inputs and the scale

with pd.option_context("display.max_colwidth", 120):
    display(rs.windows_table())

## 1 · Load the TUNE panel and build the inputs (Table 1a, Table 1b)

`research.load` reads the stored 1-minute bars of the five names, from the TUNE start through its last session, and
attaches the option inputs from the free options store. `strategy.build_inputs` then puts everything on ONE grid of
`BAR_MINUTES` bars per session, because Darts needs every input on the target's own time steps. The grid follows
the clock (09:30–09:35, 09:35–09:40, …). The steps are numbered 0, 1, 2 … across sessions; nights and weekends are
not on the grid at all.

* A **faster** input (the 1-minute bars) is SUMMARIZED into each grid bar.
* A **slower** input (daily, options) is REPEATED on every grid bar of its session: the value known before the open,
  never that day's own close.
* On the NYSE **half-days** (13:00 close) the stored "regular" panel still holds 13:00–15:59. Those grid bars stay
  on the grid but are masked: no 1-minute bar after the close enters any input, and they carry no target and no
  trade (check 8f).

**Table 1a** is a data-quality summary per name. **Table 1b** shows `SAMPLE_NAME`'s first grid bars on
`SAMPLE_DAY` exactly as the model receives them: one column per grid bar, one row per input. A NaN there reaches
Darts as 0. An option source's `has_*` flag is 0 only when that WHOLE source is missing; §3 counts the cells where
one input is missing while its flag still says 1, and the missing values of inputs that have no flag.

In [ ]:
panel = rs.load(settings, engine)  # TUNE only: data.start .. its last session, options attached
KW = dict(
    bar_minutes=BAR_MINUTES,
    horizon=HORIZON,
    options=OPTIONS,
    use_daily=USE_DAILY,
    use_market=USE_MARKET,
    vol_days=VOL_DAYS,
)
inp = SM["build_inputs"](panel, **KW)
g = inp["g"]
NAMES, INPUTS, GROUPS = inp["names"], inp["past_names"], inp["groups"]
COL = {k: i for i, k in enumerate(INPUTS)}
NAME_COLOR = {s: plots.SERIES[i] for i, s in enumerate(NAMES)}
code, sf = g["code"], g["sess_first"]
n_sess = int(code.max()) + 1
SESS = pd.DatetimeIndex(pd.to_datetime(g["date"][sf]))  # session number -> date
SESS_UTC = SESS.tz_localize("UTC")  # the index of panel.features: UTC midnight of each session
# dates as strings on time axes: plotly.js would show epoch milliseconds in the viewer's timezone
DAYS = list(SESS.strftime("%Y-%m-%d"))
after_warmup = code >= WARMUP  # grid bars the walk-forward can learn from or score
REAL = after_warmup & g["regular"]  # ... inside the real session (not after a half-day's close)
HALF = [d for d in DAYS if d in SM["HALF_DAYS"]]  # the NYSE 13:00 closes inside the panel
local = panel.close.index.tz_convert(panel.tz)
A = np.stack([inp["past"][s] for s in NAMES])  # names x grid bars x inputs: what the model reads
DAY = pd.Timestamp(SAMPLE_DAY).date()
day_rows = np.flatnonzero(g["date"] == DAY)
assert len(day_rows), f"SAMPLE_DAY {SAMPLE_DAY} is not a session of the TUNE panel"
assert SAMPLE_NAME in NAMES, f"SAMPLE_NAME must be one of {NAMES}"

group_n = {k: sum(v == k for v in GROUPS.values()) for k in dict.fromkeys(GROUPS.values())}
per_sess = pd.Series(code).value_counts().value_counts().sort_index().to_dict()
SOURCES = SM["OPTION_SET_SOURCES"][OPTIONS]  # the option sources this OPTIONS setting reads
print(
    f"1-minute panel: {len(panel.close):,} bars x {len(NAMES)} names ({', '.join(NAMES)}), "
    f"{n_sess} sessions {SESS[0]:%Y-%m-%d} -> {SESS[-1]:%Y-%m-%d} (TUNE only)"
)
print(
    f"grid: {BAR_MINUTES}-minute bars by the clock, {g['n_pos']} per full session, {len(code):,} "
    "grid bars (" + ", ".join(f"{n} sessions x {k} grid bars" for k, n in per_sess.items()) + "); "
    f"{len(HALF)} half-days ({', '.join(HALF)}) keep their {int((~g['regular']).sum())} grid bars "
    "after the 13:00 close on the grid, masked"
)
print(
    f"inputs per name and grid bar: {len(INPUTS)} past covariates ("
    + ", ".join(f"{n} {k}" for k, n in group_n.items())
    + f"), {len(inp['future_names'])} future covariates (the clock), {len(NAMES)} static "
    f"(which name) and the target's own last {P['lookback']} values"
)
print(
    f"options = {OPTIONS!r}: {len(SOURCES)} sources, one flag each "
    f"({', '.join('has_' + s for s in SOURCES) or 'none'}); option frames attached to the "
    f"panel: {inp['options_attached']}"
)
print(
    f"target: the z-scored return of the next {HORIZON} grid bar(s); usable (finite, decided in "
    f"the same session): {inp['valid'].mean():.1%} of grid bars x names"
)
print(
    f"warm-up: sessions 1-{WARMUP} ({SESS[0]:%Y-%m-%d} -> {SESS[WARMUP - 1]:%Y-%m-%d}) only build "
    f"the daily inputs and the volatility scale"
)
quality = pd.DataFrame(
    {
        "1m bars": panel.close.notna().sum(),
        "missing 1m bars %": 100 * panel.close.isna().mean(),
        "zero-volume minutes %": 100 * (panel.volume == 0).mean(),
        "median M shares / session": panel.volume.groupby(np.asarray(local.date)).sum().median()
        / 1e6,
    }
).reindex(NAMES)
print("Table 1a · data quality per name")
display(quality.round(2))

peek_rows = day_rows[:4]
peek = pd.DataFrame(
    A[NAMES.index(SAMPLE_NAME)][peek_rows].T,
    index=pd.Index(INPUTS, name="input"),
    columns=[f"step {b} (closes {rs.clock(g['pos'][b], BAR_MINUTES)})" for b in peek_rows],
)
peek.insert(0, "group", [GROUPS[k] for k in INPUTS])
print(f"Table 1b · {SAMPLE_NAME} on {SAMPLE_DAY}: the first grid bars as the model gets them")
display(peek.round(3))

## 2 · The split: which data trains, validates and tests (Table 2a, Fig 2a, Fig 2b, Table 2)

There are two levels of split, and they answer different questions. **Table 2a** prints both with this run's dates.

| level | slice | what it is | where |
|---|---|---|---|
| researcher (outer) | **TUNE** | knobs may be chosen on these forecasts; the only window loaded here | `research.WINDOWS["tune"]` |
| walk-forward | warm-up | builds the daily inputs and the volatility scale; never fitted or scored | the first `WARMUP` sessions |
| every fit (inner) | **TRAIN** | the rows the model is fitted on | the last `TRAIN_DAYS` sessions before the retrain point |
| every fit (inner) | **VALIDATION** | held out for early stopping only: tide / tft with `val_days > 0`, never gbm | the last `VAL_DAYS` of those sessions |
| every fit (inner) | **FORECAST** | out-of-sample predictions, each made from bars up to its own decision bar | from the retrain point to the next one; scored from `optuna.score_from`, the first retrain point |
| researcher (outer) | **TEST** | the knobs frozen, scored once (diagnostics notebook) | `research.WINDOWS["test"]` |
| researcher (outer) | **HOLDOUT** | locked: read once, behind `READ_HOLDOUT = True` | `research.WINDOWS["holdout"]` → last stored bar |

* **Every scored forecast is out of sample** at the inner level: the model that made it never saw its outcome.
* **TUNE is still in-sample for the researcher.** Choosing knobs on the TUNE forecasts fits the knobs to TUNE.
  For `gbm` (the default) there is no inner VALIDATION slice: it boosts a fixed `gbm_iter` rounds and is never
  early-stopped. The TUNE forecasts play the role of a validation set, and TEST is the test set.
* **TEST freezes the knobs, not the weights.** The walk-forward keeps retraining on its trailing `TRAIN_DAYS`
  sessions, exactly as it would live. From late March on, its training windows lie entirely inside TEST. That
  is allowed, because each fit uses only the past.

**How to read Fig 2a**, the whole timeline:
* One row per data source; each option source carries the name of its `has_*` flag. Dark blue is data used in this
  notebook (TUNE). Light blue is stored but not used here. Its prices are never loaded: only the first and last date
  are read. Its option days are read by `research.load` only to be dropped.
* The walk-forward row shows the warm-up, the first training window and the forecasts. It continues through TEST
  (scored once) and HOLDOUT (locked). The dotted line is `optuna.score_from`.
* The bottom row is the one fit that forecasts `SAMPLE_DAY`.

**Fig 2b and Table 2** show the fits around `SAMPLE_DAY` on the grid's own clock. Sessions sit side by side,
because nights and weekends do not exist for the model. The TRAIN window slides forward one retrain step at a time.

In [ ]:
pts = SM["schedule"](g, train_days=TRAIN_DAYS, retrain_minutes=RETRAIN_MINUTES, warmup=WARMUP)
assert len(pts), "the TUNE panel is too short for the warm-up plus one training window"
deep = MODEL in rs.SM["DEEP"]
v_days = VAL_DAYS if deep else 0  # walk_forward holds out a validation slice for tide / tft only
CADENCE = (
    "trained once (a static model)"
    if RETRAIN_MINUTES == 0
    else f"retrained every {RETRAIN_MINUTES} trading minutes"
)
F = panel.features or {}
ONE = pd.Timedelta(days=1)
W = {k: (pd.Timestamp(a), pd.Timestamp(b) if b else None) for k, (a, b) in rs.WINDOWS.items()}
C_TRAIN, C_VAL, C_FC = plots.SERIES[1], plots.SERIES[2], plots.SERIES[4]
C_DATA, C_STORED = plots.SERIES[0], "#cfe0f4"
SLICE_COLORS = (C_TRAIN, C_VAL, C_FC)  # TRAIN, VALIDATION, FORECAST (rs.slices)
score_from = pd.Timestamp(strat.optuna["score_from"])
s0 = int(code[pts[0]])  # the first retrain point's session

with engine.connect() as conn:  # dates only: no price outside TUNE is read
    db = pd.read_sql(
        text(
            "SELECT min(ts) AS first, max(ts) AS last FROM prices_ohlcv "
            "WHERE bar = '1m' AND symbol = ANY(:s) AND rth"
        ),
        conn,
        params={"s": NAMES},
    ).iloc[0]
SRC = {
    "1-minute bars": (None, (rs.ny_day(db["first"], panel.tz), rs.ny_day(db["last"], panel.tz)), "")
}
for src, names in SM["OPTION_SOURCES"].items():  # one row per option source, with its flag
    SRC[f"options: {SOURCE_LABEL[src]}"] = (
        ["opt_" + n for n in names],
        rs.store_span(STORE_OF[src]),
        f"has_{src} · ",
    )
last_stored = max(s[1] for _, s, _ in SRC.values() if s[1] is not None)
tune_end = SESS[-1] + ONE

# --- Table 2a: the split, with this run's dates ---------------------------------------------------
outcome_ok = np.zeros(inp["valid"].shape, bool)  # decision s is scored on the target at s + HORIZON
outcome_ok[:-HORIZON] = inp["valid"][HORIZON:]
scored = (np.arange(len(code)) >= pts[0])[:, None] & (g["left"] >= HORIZON)[:, None] & outcome_ok
split = pd.DataFrame(
    [
        {
            "slice": "TUNE (researcher)",
            "which sessions": 'research.WINDOWS["tune"]',
            "this run": f"{rs.ymd(W['tune'][0])} -> {rs.ymd(W['tune'][1])}: {n_sess} sessions, the "
            "only window loaded here",
            "use": "knobs may be chosen on its forecasts",
        },
        {
            "slice": "warm-up (walk-forward)",
            "which sessions": f"the first WARMUP = {WARMUP}",
            "this run": f"{DAYS[0]} -> {DAYS[WARMUP - 1]}",
            "use": "builds the daily inputs and the volatility scale; never fitted or scored",
        },
        {
            "slice": "TRAIN (every fit)",
            "which sessions": f"the last TRAIN_DAYS = {TRAIN_DAYS} before each retrain point"
            + (", minus VALIDATION" if v_days else ""),
            "this run": f"1st fit: {DAYS[s0 - TRAIN_DAYS]} -> {DAYS[s0 - v_days - 1]}; then "
            f"{CADENCE}",
            "use": "fitted (weight 0 on unusable targets)",
        },
        {
            "slice": "VALIDATION (every fit)",
            "which sessions": "tide / tft: the last VAL_DAYS sessions of TRAIN",
            "this run": f"1st fit: {DAYS[s0 - v_days]} -> {DAYS[s0 - 1]} ({v_days} sessions)"
            if v_days
            else f"none: {MODEL} is never early-stopped",
            "use": "early stopping only, never fitted",
        },
        {
            "slice": "FORECAST (every fit)",
            "which sessions": "from each retrain point to the next",
            "this run": f"{DAYS[s0]} -> {DAYS[-1]}: {len(pts)} fits, {int(scored.sum()):,} "
            "scored forecasts",
            "use": f"out of sample; scored from optuna.score_from = {score_from:%Y-%m-%d}",
        },
        {
            "slice": "TEST (researcher)",
            "which sessions": 'research.WINDOWS["test"]',
            "this run": f"{rs.ymd(W['test'][0])} -> {rs.ymd(W['test'][1])}: not loaded here",
            "use": "the frozen config, scored ONCE; the walk-forward continues into it warm",
        },
        {
            "slice": "HOLDOUT (researcher)",
            "which sessions": 'research.WINDOWS["holdout"]',
            "this run": f"{rs.ymd(W['holdout'][0])} -> {rs.ymd(last_stored)} (last stored): "
            "not loaded",
            "use": "locked: read once, behind READ_HOLDOUT = True",
        },
    ]
).set_index("slice")
print("Table 2a · the split: which data trains, validates and tests")
with pd.option_context("display.max_colwidth", 100):
    display(split)

fig = go.Figure()
for name, (cols, (s_first, s_last), flag) in SRC.items():
    used = cols is None or any(c in COL for c in cols)
    span = f"stored {s_first:%Y-%m-%d} -> {s_last:%Y-%m-%d}" if s_first is not None else "none"
    row = f"{name}<br><sup>{flag}{span}{'' if used else ' · not an input with these knobs'}</sup>"
    for a, b in rs.runs(rs.loaded(F, cols, SESS_UTC)):
        charts.bar(fig, row, SESS[a], SESS[b] + ONE, C_DATA, "used here (TUNE)")
    if s_last is not None and s_last >= tune_end:
        charts.bar(fig, row, tune_end, s_last + ONE, C_STORED, "stored, NOT used here")

wf = "walk-forward<br><sup>a fresh fit at every retrain point</sup>"
charts.bar(
    fig, wf, SESS[0], SESS[WARMUP], plots._MUTED, "warm-up (inputs and scale only)", "warm-up"
)
charts.bar(
    fig, wf, SESS[s0 - TRAIN_DAYS], SESS[s0 - v_days], C_TRAIN, "TRAIN (fitted)", "1st TRAIN"
)
if v_days:
    charts.bar(fig, wf, SESS[s0 - v_days], SESS[s0], C_VAL, "VALIDATION (early stopping)")
n_fits = f"{len(pts)} fit{'s' if len(pts) > 1 else ''}, scored"
charts.bar(fig, wf, SESS[s0], tune_end, C_FC, "FORECAST (out of sample)", n_fits)
charts.bar(
    fig,
    wf,
    W["test"][0],
    W["test"][1] + ONE,
    C_FC,
    "TEST: knobs frozen, scored ONCE",
    "scored once",
    opacity=0.45,
    pattern="/",
)
charts.bar(
    fig,
    wf,
    W["holdout"][0],
    last_stored + ONE,
    plots._MUTED,
    "HOLDOUT: locked",
    "locked",
    opacity=0.35,
    pattern="x",
)
kf = max(int(np.searchsorted(pts, day_rows[0], side="right")) - 1, 0)  # the fit forecasting DAY
T = int(pts[kf])
s_k = int(code[T])
T_next = int(pts[kf + 1]) if kf + 1 < len(pts) else len(code)
lo_k = max(0, s_k - TRAIN_DAYS)
covers = day_rows[0] >= pts[0]  # False when SAMPLE_DAY lies before the first fit
which = (
    f"the fit that forecasts {SAMPLE_DAY}" if covers else "the first fit (SAMPLE_DAY is earlier)"
)
one = f"one retrain point<br><sup>{which}</sup>"
charts.bar(fig, one, SESS[lo_k], SESS[s_k - v_days], C_TRAIN, "TRAIN (fitted)", "TRAIN")
if v_days:
    charts.bar(fig, one, SESS[s_k - v_days], SESS[s_k], C_VAL, "VALIDATION (early stopping)")
charts.bar(fig, one, SESS[s_k], SESS[int(code[T_next - 1])] + ONE, C_FC, "FORECAST (out of sample)")

shade = {"tune": plots.SERIES[0], "test": plots.SERIES[2], "holdout": plots._MUTED}
for w, (a, b) in W.items():  # shapes after the traces (CLAUDE.md: plotly drops them otherwise)
    b = (b if b is not None else last_stored) + ONE
    fig.add_vrect(x0=a, x1=b, fillcolor=shade[w], opacity=0.08, line_width=0, layer="below")
    fig.add_annotation(
        x=a,
        y=1.0,
        yref="paper",
        xanchor="left",
        yanchor="bottom",
        showarrow=False,
        text=f"<b>{w.upper()}</b><br>{a:%Y-%m-%d} → {b - ONE:%Y-%m-%d}",
        font={"size": 11, "color": plots._INK},
    )
fig.add_shape(
    type="line",
    x0=score_from,
    x1=score_from,
    y0=0,
    y1=1,
    yref="paper",
    line={"color": plots._INK, "dash": "dot", "width": 1},
)
fig.add_annotation(  # beside the window labels: lower down it would cover the row labels
    x=score_from,
    y=1.0,
    yref="paper",
    xanchor="left",
    yanchor="bottom",
    showarrow=False,
    text=f"<b>optuna.score_from</b><br>{score_from:%Y-%m-%d}: the first fitted model",
    font={"size": 11, "color": plots._INK},
)
fig.add_annotation(
    x=SESS[s_k] + ONE / 2,
    y=one,
    text=f"FORECAST {SESS[s_k]:%Y-%m-%d}",
    showarrow=True,
    arrowhead=2,
    arrowcolor=C_FC,
    ax=60,
    ay=0,
    xanchor="left",
    font={"size": 10, "color": C_FC},
)
order = list(dict.fromkeys(t.y[0] for t in fig.data))
fig.update_yaxes(categoryorder="array", categoryarray=order[::-1])
fig.update_xaxes(
    type="date", range=[SESS[0] - pd.Timedelta(days=12), last_stored + pd.Timedelta(days=12)]
)
fig.update_layout(barmode="overlay", bargap=0.35, uniformtext={"minsize": 9, "mode": "hide"})
plots._base(
    fig, title="Fig 2a · the split: what is loaded, fitted, validated and forecast", height=620
)
fig.update_layout(margin={"l": 230, "t": 100, "b": 60})
charts.below(fig, -0.1)
fig.show()

# --- Fig 2b / Table 2: the fits around SAMPLE_DAY on the grid's own clock -------------------------
n_pos = g["n_pos"]

fig = go.Figure()
info, rows2 = [], []
show = list(range(max(0, kf - 1), min(len(pts), kf + 2)))
cases = [
    (
        i,
        v_days,
        f"fit {i + 1} of {len(pts)}<br><sup>retrain {rs.when(int(pts[i]), g, BAR_MINUTES)}</sup>",
    )
    for i in show
]
if not v_days:  # show what a tide / tft validation slice would look like
    cases.append(
        (
            kf,
            5,
            f"example: tide / tft, val_days = 5<br><sup>retrain {rs.when(T, g, BAR_MINUTES)}</sup>",
        )
    )
for i, val, row in cases:
    t, t_next = int(pts[i]), int(pts[i + 1]) if i + 1 < len(pts) else len(code)
    rows2.append(row)
    for b0, b1, color, name in rs.slices(t, t_next, val, g, TRAIN_DAYS, SLICE_COLORS):
        if b1 <= b0:
            continue
        x0, w = rs.x_of(b0, b1, g)
        n_s = (b1 - b0) / n_pos
        word = name.split(" ")[0]
        label = f"{word} · {n_s:.0f} sessions" if n_s >= 1 else f"{word} · {b1 - b0} grid bars"
        fig.add_trace(
            go.Bar(
                y=[row],
                x=[w],
                base=[x0],
                orientation="h",
                marker={"color": color, "opacity": 1.0 if row.startswith("fit") else 0.55},
                text=[label],
                textposition="inside",
                insidetextanchor="middle",
                name=name,
                legendgroup=name,
                showlegend=name not in {tr.name for tr in fig.data},
                hovertemplate=(
                    f"<b>{name}</b><br>{g['date'][b0]} "
                    f"{rs.clock(g['pos'][b0], BAR_MINUTES, end=False)} -> "
                    f"{g['date'][b1 - 1]} {rs.clock(g['pos'][b1 - 1], BAR_MINUTES)}"
                    "<extra></extra>"
                ),
            )
        )
    if row.startswith("fit"):
        lo, fit_hi = rs.slices(t, t_next, val, g, TRAIN_DAYS, SLICE_COLORS)[0][:2]
        info.append(
            {
                "fit": f"{i + 1} of {len(pts)}",
                "retrain": rs.when(t, g, BAR_MINUTES),
                "TRAIN sessions": f"{g['date'][lo]} -> {g['date'][fit_hi - 1]} "
                f"({(fit_hi - lo) / n_pos:.0f})",
                "VALIDATION sessions": f"{g['date'][fit_hi]} -> {g['date'][t - 1]} "
                f"({(t - fit_hi) / n_pos:.0f})"
                if t > fit_hi
                else "none",
                "FORECAST decision bars": f"{g['date'][t]} {rs.clock(g['pos'][t], BAR_MINUTES)} -> "
                f"{g['date'][t_next - 1]} {rs.clock(g['pos'][t_next - 1], BAR_MINUTES)} "
                f"({t_next - t})",
                "usable training targets": int(inp["valid"][lo:fit_hi].sum()),
            }
        )
x_lo = int(code[int(sf[max(0, int(code[pts[show[0]]]) - TRAIN_DAYS)])]) - 1
x_hi = int(code[(int(pts[show[-1] + 1]) if show[-1] + 1 < len(pts) else len(code)) - 1]) + 2
ticks = [t for t in range(x_lo + 1, x_hi, 5) if 0 <= t < n_sess]
fig.add_vrect(
    x0=code[day_rows[0]], x1=code[day_rows[0]] + 1, fillcolor=plots._INK, opacity=0.06, line_width=0
)
fig.update_xaxes(
    range=[x_lo, x_hi],
    tickvals=ticks,
    ticktext=[f"{SESS[t]:%m-%d}" for t in ticks],
    title_text="sessions side by side (a tick = that session's open); the shaded day is SAMPLE_DAY",
)
fig.update_yaxes(categoryorder="array", categoryarray=rows2[::-1])
fig.update_layout(barmode="overlay", bargap=0.3, uniformtext={"minsize": 9, "mode": "hide"})
plots._base(
    fig,
    title=f"Fig 2b · the walk-forward around {SAMPLE_DAY}: {MODEL}, {CADENCE}, "
    f"TRAIN = {TRAIN_DAYS} sessions",
    height=360,
)
fig.update_layout(margin={"l": 230})
charts.below(fig, -0.3)
fig.show()
print("Table 2 · the fits around SAMPLE_DAY (walk_forward's own slicing)")
with pd.option_context("display.max_colwidth", 120):
    display(pd.DataFrame(info).set_index("fit"))
print(
    f"A forecast's outcome must end inside its session, so the last {HORIZON} decision bar(s) of "
    "each session are forecast but never scored or traded."
)
print(
    f"Scored out-of-sample forecasts in TUNE: {int(scored.sum()):,} (decision bars from the first "
    f"retrain point on, over {n_sess - s0} sessions x {len(NAMES)} names); the model notebook "
    "scores exactly these."
)
print(
    f"{len(pts)} retrain point{'s' if len(pts) > 1 else ''} in TUNE, {CADENCE}. The first, "
    f"{rs.when(int(pts[0]), g, BAR_MINUTES)}, makes the first scored forecasts; "
    f"optuna.score_from = {strat.optuna['score_from']}."
)
if not deep:
    print(
        f"{MODEL}: no VALIDATION slice, every TRAIN session is fitted (val_days is ignored). "
        "tide / tft with val_days > 0 hold the last val_days sessions out for early stopping "
        "(the faded example row)."
    )

## 3 · The input inventory (Table 3, Table 3b)

One row per model input:
* **source, resolution, known at**: where the input comes from, how often it changes, and the moment its value is
  known.
* **flag**: the `has_*` input that marks it missing, if it has one.
* **gbm lags**: read off the model `make_model` actually builds. `s` is the decision grid bar, and `s-5 .. s (6)`
  means the six latest values. Intraday and market inputs get `cov_lags` lags. The daily, option and flag inputs
  get only their latest value: they are constant within a session, so more lags would repeat the same number.
  TiDE / TFT instead read every past input over the whole `lookback` window.
* **% missing per name**: on the real-session grid bars after the warm-up, in TUNE. A missing value enters Darts as
  0, because its tabular models refuse NaN.

**One `has_*` flag per option source.** `has_dolt_iv` covers the DoltHub chain (atm_iv, skew, term, d_atm_iv),
`has_dolt_hv` the DoltHub vol history (iv_hv), `has_occ` OCC (cust_pc, vol_surge) and `has_oi` MarketData (oi_pc,
vol_pc, d_oi, gex). A flag is 1 when ANY input of its source has data. So it tells "zero" from "no data" whenever a
whole source is missing: MarketData before October 2025, or a DoltHub day missing for one name. It cannot mark one
input missing inside a present source, and the daily inputs have no flag at all. **Table 3b** counts those cells:
they reach the model as a plain 0. On the default run it has one row: `opt_d_oi` on 2025-10-06, MarketData's first
session, which has no previous day to take the change from.

What to look for: inputs that are mostly missing, and inputs whose timing you would not have guessed. The
MarketData open-interest set only starts in October 2025, so the first training windows never see it.

In [ ]:
DESC = {
    "ret": "the grid bar's return (z)",
    "rv": "realized vol of its 1-minute returns (z)",
    "range": "log high / low of the grid bar (z)",
    "clv": "close location in the bar's range (-1 low .. +1 high)",
    "flow": "signed-volume flow: up-minute minus down-minute volume, over volume",
    "rvol": "log volume vs the same grid bar of the previous 10 sessions",
    "ret_open": "return since the open (z)",
    "vwap_dist": "close vs the session VWAP (z)",
    "mkt": "the other names' average return (z)",
    "prev_ret": "previous session's return / 20-day vol",
    "ret5": "5-session return / (20-day vol x sqrt 5)",
    "ret20": "20-session return / (20-day vol x sqrt 20)",
    "vol20": "log 20-day vol of daily returns",
    "pos20": "previous close in the 20-day high-low range (0..1)",
    "prev_clv": "previous session's close location (-1..+1)",
    "prev_rvol": "previous session's volume vs its 20-day mean (log)",
    "gap": "today's first open vs the previous close / 20-day vol",
    "opt_atm_iv": "30-day at-the-money implied vol",
    "opt_skew": "25-delta put IV minus 25-delta call IV",
    "opt_term": "near-expiry ATM IV minus 30-day ATM IV",
    "opt_d_atm_iv": "change of ATM IV from the previous stored day (<= 4 days back)",
    "opt_iv_hv": "implied minus historical vol",
    "opt_cust_pc": "OCC customer put / call volume (log)",
    "opt_vol_surge": "option volume vs its previous 20-day mean (log)",
    "opt_oi_pc": "put / call open interest (log)",
    "opt_vol_pc": "put / call option volume (log)",
    "opt_d_oi": "change of total open interest (log)",
    "opt_gex": "dealer gamma imbalance (-1..+1, model-based)",
    **{f"has_{s}": f"1 if any {SOURCE_LABEL[s]} input has data" for s in SM["OPTION_SOURCES"]},
}
try:  # the lags of the model make_model really builds
    _m = SM["make_model"](
        "gbm",
        horizon=HORIZON,
        lookback=P["lookback"],
        cov_lags=P["cov_lags"],
        past_names=INPUTS,
        groups=GROUPS,
    )
    LAGS = {k: len(v) for k, v in _m.component_lags["past"].items()}
    T_LAGS = len(_m.lags["target"])
except ImportError:  # Darts is an optional extra: make_model's rule
    LAGS = {k: P["cov_lags"] if GROUPS[k] in ("intraday", "market") else 1 for k in INPUTS}
    T_LAGS = P["lookback"]

rows3 = []
for k in INPUTS:
    grp = GROUPS[k]
    if grp in ("intraday", "market"):
        src, res, known = (
            "1-minute bars",
            f"1m into each {BAR_MINUTES}m grid bar",
            "grid bar's close",
        )
    elif grp == "daily":
        src, res = "1-minute bars, per session", "1 per session, repeated"
        known = "today's first open" if k == "gap" else "previous close"
    elif grp == "options":
        src, res = f"{SOURCE_LABEL[SOURCE_OF[k]]} (EOD)", "1 per day (EOD), repeated"
        known = "end of the last stored day before the session (<= 5 days old)"
    else:
        src, res, known = "derived", "as its source", "with its source"
    rows3.append(
        {
            "input": k,
            "group": grp,
            "what": DESC.get(k, ""),
            "source": src,
            "resolution": res,
            "known at": known,
            "flag": f"has_{SOURCE_OF[k]}" if k in SOURCE_OF else "-",
            "gbm lags": rs.lag_txt(LAGS[k]),
            **rs.missing(A[:, REAL, COL[k]], NAMES),
        }
    )
rows3.append(
    {
        "input": "(target lags)",
        "group": "target",
        "what": "the z-scored grid return itself",
        "source": "1-minute bars",
        "resolution": "per grid bar",
        "known at": "grid bar's close",
        "flag": "-",
        "gbm lags": rs.lag_txt(T_LAGS),
        **rs.missing(inp["y"][REAL].T, NAMES),
    }
)
rows3.append(
    {
        "input": ", ".join(inp["future_names"]),
        "group": "future",
        "what": "time of day (sin, cos, fraction), weekday, monthly-expiry week",
        "source": "calendar",
        "resolution": "per grid bar",
        "known at": "in advance",
        "flag": "-",
        "gbm lags": f"the predicted bar s+{HORIZON}",
        **rs.missing(np.zeros((len(NAMES), 1)), NAMES),
    }
)
rows3.append(
    {
        "input": "is_<name>",
        "group": "static",
        "what": "which name (one-hot): one global model",
        "source": "panel",
        "resolution": "1 per name",
        "known at": "always",
        "flag": "-",
        "gbm lags": "-",
        **rs.missing(np.zeros((len(NAMES), 1)), NAMES),
    }
)
inventory = pd.DataFrame(rows3).set_index("input")
print("Table 3 · every input the model reads")
with pd.option_context("display.max_colwidth", 70):
    display(inventory.round(1))
print(
    f"tide / tft read every past input over the whole lookback window of {P['lookback']} grid "
    f"bars (input_chunk_length) instead of per-input lags"
)
for src in SOURCES:  # each option source's flag: when its data starts, how often it is there
    has = A[:, :, COL[f"has_{src}"]] > 0.5  # names x grid bars
    first = SESS[int(code[np.flatnonzero(has.any(axis=0))[0]])] if has.any() else None
    print(
        f"has_{src} ({SOURCE_LABEL[src]}): first TUNE session with data {first:%Y-%m-%d}; 1 on "
        f"{has[:, REAL].mean():.1%} of the grid bars after the warm-up"
        if first is not None
        else f"has_{src} ({SOURCE_LABEL[src]}): no data in TUNE"
    )

# Table 3b: a missing value that no flag marks reaches the model as a plain 0
rows3b = []
for k in INPUTS:
    if GROUPS[k] == "flag":
        continue
    flag = f"has_{SOURCE_OF[k]}" if k in SOURCE_OF else None
    x = A[:, REAL, COL[k]]
    on = A[:, REAL, COL[flag]] > 0.5 if flag else np.ones(x.shape, bool)
    unmarked = np.isnan(x) & on  # names x grid bars
    if unmarked.any():
        days_ = sorted({DAYS[int(c)] for c in code[REAL][unmarked.any(axis=0)]})
        rows3b.append(
            {
                "input": k,
                "its flag": flag or "none",
                "flag = 1 %": 100 * on.mean() if flag else np.nan,
                "missing %": 100 * np.isnan(x).mean(),
                "missing, unmarked %": 100 * unmarked.mean(),
                **{f"{s} %": 100 * unmarked[j].mean() for j, s in enumerate(NAMES)},
                "sessions": ", ".join(days_[:5])
                + (f" ... ({len(days_)} sessions)" if len(days_) > 5 else ""),
            }
        )
print("Table 3b · missing values no flag marks: they reach the model as a plain 0")
if rows3b:
    with pd.option_context("display.max_colwidth", 80):
        display(pd.DataFrame(rows3b).set_index("input").round(2))
else:
    print("none with these knobs")

## 4 · The target (Table 4, Fig 4a, Fig 4b, Fig 4c)

The model predicts `y`: each grid bar's log return divided by the name's **volatility scale**. The scale is the RMS
grid-bar return over the previous `VOL_DAYS` sessions. It is causal: today's bars never enter today's scale. This
puts a calm name (CSCO) and a wild one (COIN) on one footing, so one global model can learn from all five. The label
for a decision at grid bar `s` is the sum of `y` over the next `HORIZON` bars, `s+1 .. s+HORIZON`. The first grid
bar of a session starts at that session's open, so the overnight gap is never part of the target.

How to read it:
* **Table 4**: a well-scaled target has std ≈ 1. Heavy tails show as excess kurtosis, and as far more |z| > 3 bars
  than a normal's 0.27%. The lag-1 autocorrelation is the simplest "signal" there is. Negative means the next bar
  tends to reverse this one (the bid-ask bounce), positive that it tends to continue it, and near zero that one
  bar's return says little about the next (§9 measures `ret` as an input).
* **Fig 4a**: the z-score distribution per name, on a log scale, against a standard normal (dotted).
* **Fig 4b**: the scale over time, in basis points. It moves slowly, so after a jump in volatility it lags by up to
  `VOL_DAYS` sessions and the z-scores run large meanwhile.
* **Fig 4c**: the z-scores are still bigger at the open and the close than at midday. The scale is one number per
  session, so the intraday U-shape survives the scaling. The clock inputs (future covariates) exist for exactly
  this.

**Which targets the model is fitted on.** A target counts (`valid`) only when it is finite and its decision bar,
`HORIZON` grid bars earlier, lies in the same session. So each session's first `HORIZON` grid bars, decided at the
previous session's close, get weight 0 in every fit: never fitted, scored or traded (check 8d). That matters,
because the opening bar is the biggest of the day (Fig 4c). The line under Fig 4c gives its share of the target's
squared size, and the share of the fitted targets decided inside the first `skip_open` minutes: those are fitted
and scored, but never traded.

In [ ]:
y = inp["y"]
scale_bp = inp["scale"][sf] * 1e4  # one value per session and name
pair = after_warmup & ~g["first"]  # (grid bar, previous grid bar) pairs inside one session
rows4 = []
for j, s in enumerate(NAMES):
    v = pd.Series(y[after_warmup, j]).dropna()
    rows4.append(
        {
            "name": s,
            "grid bars": len(v),
            "mean": v.mean(),
            "std": v.std(),
            "skew": v.skew(),
            "excess kurtosis": v.kurt(),
            "|z| > 3 %": 100 * (v.abs() > 3).mean(),
            "|z| > 5 %": 100 * (v.abs() > 5).mean(),
            "lag-1 autocorr": pd.Series(y[pair, j]).corr(pd.Series(y[np.flatnonzero(pair) - 1, j])),
            "median scale bp": np.nanmedian(scale_bp[WARMUP:, j]),
        }
    )
print(f"Table 4 · the target y per name (TUNE after the warm-up, {BAR_MINUTES}-minute grid)")
display(pd.DataFrame(rows4).set_index("name").round(3))
print("a normal: std 1, excess kurtosis 0, |z| > 3 in 0.27% and |z| > 5 in 0.00006% of bars")

edges = np.arange(-8.0, 8.01, 0.25)
mid = (edges[:-1] + edges[1:]) / 2
fig = go.Figure()
for j, s in enumerate(NAMES):
    v = y[after_warmup, j]
    dens, _ = np.histogram(v[np.isfinite(v)], bins=edges, density=True)
    fig.add_trace(
        go.Scatter(
            x=mid,
            y=np.where(dens > 0, dens, np.nan),
            mode="lines",
            line={"shape": "hvh", "color": NAME_COLOR[s], "width": 1.5},
            name=s,
        )
    )
fig.add_trace(
    go.Scatter(
        x=mid,
        y=np.exp(-(mid**2) / 2) / np.sqrt(2 * np.pi),
        mode="lines",
        line={"color": plots._INK, "dash": "dot", "width": 1},
        name="standard normal",
    )
)
fig.update_yaxes(type="log", range=[-4.5, 0], title_text="density (log scale)")
fig.update_xaxes(title_text="z-scored grid-bar return")
plots._base(
    fig, title=f"Fig 4a · the target: z-scored {BAR_MINUTES}-minute returns per name", height=400
)
charts.below(fig, -0.22)
fig.show()

fig = go.Figure()
for j, s in enumerate(NAMES):
    fig.add_trace(
        go.Scatter(
            x=DAYS,
            y=scale_bp[:, j],
            mode="lines",
            name=s,
            line={"color": NAME_COLOR[s], "width": 1.5},
        )
    )
fig.add_vrect(
    x0=SESS[0], x1=SESS[WARMUP], fillcolor=plots._MUTED, opacity=0.12, line_width=0, layer="below"
)
fig.update_xaxes(type="date")
fig.update_yaxes(title_text=f"typical {BAR_MINUTES}-minute move (bp)")
plots._base(
    fig,
    title=f"Fig 4b · the volatility scale: RMS grid-bar return of the previous {VOL_DAYS} sessions "
    "(grey = warm-up)",
    height=380,
)
charts.below(fig, -0.15)
fig.show()

rms = (
    pd.DataFrame(y[after_warmup] ** 2, columns=NAMES).groupby(g["pos"][after_warmup]).mean() ** 0.5
)
fig = go.Figure()
for s in NAMES:
    fig.add_trace(
        go.Scatter(
            x=570 + (rms.index.to_numpy() + 1) * BAR_MINUTES,
            y=rms[s],
            mode="lines",
            name=s,
            line={"color": NAME_COLOR[s], "width": 1.5},
        )
    )
fig.add_hline(y=1.0, line={"color": plots._INK, "dash": "dot", "width": 1})
ticks_m = list(range(600, 961, 30))
fig.update_xaxes(
    title_text="grid bar (New York time of its close)",
    tickvals=ticks_m,
    ticktext=[f"{m // 60:02d}:{m % 60:02d}" for m in ticks_m],
)
fig.update_yaxes(title_text="RMS of z")
plots._base(
    fig, title="Fig 4c · the size of the target by time of day (1 = the average bar)", height=360
)
charts.below(fig, -0.3)
fig.show()
# each target's weight in a squared-error fit (TUNE after the warm-up)
tg = inp["target"][after_warmup]
sq = np.where(np.isfinite(tg), tg, 0.0) ** 2
fit_w = inp["valid"][after_warmup]  # walk_forward's sample weight is 0 where this is False
dec = g["pos"][after_warmup] - HORIZON  # the target's decision bar, as a position in its session
prev = dec < 0  # decided at the previous session's close
untraded = fit_w & ((dec >= 0) & ((dec + 1) * BAR_MINUTES < P["skip_open"]))[:, None]
print(
    f"targets decided at the previous session's close (each session's first {HORIZON} grid "
    f"bar(s)) are {HORIZON} in {g['n_pos']} but carry {sq[prev].sum() / sq.sum():.0%} of the "
    f"target's squared size after the warm-up; valid is False on {1 - fit_w[prev].mean():.0%} of "
    f"them, so they get weight 0 in every fit. Of the fitted targets, those decided in the first "
    f"skip_open = {P['skip_open']} minutes (fitted and scored, never traded) carry "
    f"{sq[untraded].sum() / sq[fit_w].sum():.0%} of the squared size."
)

## 5 · A sample day (Fig 5a, Fig 5b)

**Fig 5a**: `SAMPLE_NAME` on `SAMPLE_DAY`.
* The top panel is the 1-minute close, each minute drawn at its close, with the session VWAP.
* Below it are five intraday inputs, one bar per grid bar. The thin vertical lines are the grid.
* Everything inside a grid bar is summarized into one row, known at the grid bar's right edge (its close). A
  decision there fills at the next minute's open.

**Fig 5b**: three sessions on the model's own time axis, the grid step (0, 1, 2 … with no nights).
* An intraday input changes every bar.
* A daily input (`gap`, `prev_ret`) and an option input (`opt_atm_iv`) are flat within a session and step at the
  session boundary.
* The diamonds are the raw end-of-day implied vol, re-derived from the options store for that day's close. The
  arrow shows where the model first sees it: the NEXT session's first grid bar.

On the default sample (MU, 2025-12-18, the day after its earnings report) the gap appears on the day itself, known
at the open. The implied-vol drop of that day reaches the model only on 2025-12-19.

In [ ]:
j = NAMES.index(SAMPLE_NAME)
k_day = int(code[day_rows[0]])
m1 = g["codes_1m"] == k_day
t1 = local[m1].tz_localize(None) + pd.Timedelta(minutes=1)  # each minute drawn at its close
c1 = panel.close[SAMPLE_NAME].to_numpy()[m1]
tp = ((panel.high + panel.low + panel.close)[SAMPLE_NAME].to_numpy() / 3)[m1]
v1 = np.nan_to_num(panel.volume[SAMPLE_NAME].to_numpy()[m1])
vwap = np.cumsum(np.nan_to_num(tp * v1)) / np.maximum(np.cumsum(v1), 1)
t_end = local[g["end"][day_rows]].tz_localize(None) + pd.Timedelta(minutes=1)  # grid-bar closes
t_mid = t_end - pd.Timedelta(minutes=BAR_MINUTES / 2)

SHOW5 = [
    ("ret", "ret · the grid bar's return (z)"),
    ("rv", "rv · realized vol of its 1-minute returns (z)"),
    ("rvol", "rvol · volume vs the same grid bar of the previous 10 sessions (log)"),
    ("flow", "flow · signed-volume flow (-1 all down-minutes .. +1 all up)"),
    ("vwap_dist", "vwap_dist · close vs the session VWAP (z)"),
]
SHOW5 = [(k, t) for k, t in SHOW5 if k in COL]
fig = make_subplots(
    rows=1 + len(SHOW5),
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.035,
    row_heights=[0.3] + [0.7 / len(SHOW5)] * len(SHOW5),
    subplot_titles=[f"{SAMPLE_NAME}: 1-minute close, session VWAP, grid-bar closes (dots)"]
    + [t for _, t in SHOW5],
)
fig.add_trace(
    go.Scatter(
        x=rs.iso(t1),
        y=c1,
        mode="lines",
        name="1-minute close",
        line={"color": plots.SERIES[0], "width": 1.2},
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=rs.iso(t1),
        y=vwap,
        mode="lines",
        name="session VWAP",
        line={"color": plots._MUTED, "dash": "dash", "width": 1},
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=rs.iso(t_end),
        y=panel.close[SAMPLE_NAME].to_numpy()[g["end"][day_rows]],
        mode="markers",
        name="grid-bar close: a decision point",
        marker={"color": plots._INK, "size": 4},
    ),
    row=1,
    col=1,
)
for r, (k, _) in enumerate(SHOW5, start=2):
    vals = A[j, day_rows, COL[k]]
    signed = k in ("ret", "flow", "vwap_dist", "rvol")
    fig.add_trace(
        go.Bar(
            x=rs.iso(t_mid),
            y=vals,
            width=BAR_MINUTES * 60_000 * 0.85,
            marker_color=list(np.where(vals >= 0, POS_C, NEG_C)) if signed else plots.SERIES[4],
            name=k,
            showlegend=False,
            hovertemplate=f"{k} %{{y:.2f}}<extra></extra>",
        ),
        row=r,
        col=1,
    )
fig.update_xaxes(
    type="date",
    tickformat="%H:%M",
    dtick=30 * 60_000,
    minor={"dtick": BAR_MINUTES * 60_000, "showgrid": True, "gridcolor": "#efeee9"},
)
plots._base(
    fig,
    title=f"Fig 5a · {SAMPLE_NAME} on {SAMPLE_DAY}: the 1-minute bars and the intraday inputs per "
    f"{BAR_MINUTES}-minute grid bar",
    height=980,
)
fig.update_annotations(font_size=11)
charts.below(fig, -0.04)
fig.show()

In [ ]:
# The raw end-of-day option features, re-derived from the options store's own files (days up to the
# TUNE end only) with the burst_scan/options_free loaders. This is independent of research.py's
# "attach to the next session" step; checks 8c, 8c2 and 8i reuse it.
lo_day, hi_day = f"{SESS[0] - pd.Timedelta(days=150):%Y-%m-%d}", f"{SESS[-1]:%Y-%m-%d}"
# 8c2 reuses the chains
dolt_raw = rs.sessions_only(rs.read_store(STORE_OF["dolt_iv"], lo_day, hi_day), SESS)
md_raw = rs.sessions_only(rs.read_store(STORE_OF["oi"], lo_day, hi_day), SESS)
iv = of.iv_features(dolt_raw).sort_values(["underlying", "date"])  # junk quotes dropped
iv["d_atm_iv"] = iv.groupby("underlying")["atm_iv"].diff().where(rs.gap_days(iv) <= 4)  # no gaps
hv = rs.sessions_only(rs.read_store(STORE_OF["dolt_hv"], lo_day, hi_day), SESS)
hv = hv.rename(columns={"act_symbol": "underlying"})
hv["iv_hv"] = hv["iv_current"].astype(float) - hv["hv_current"].astype(float)
occ = of.occ_features(rs.sessions_only(rs.read_store(STORE_OF["occ"], lo_day, hi_day), SESS))
mdc = of.md_features(md_raw, iv)  # sorted by underlying, date
mdc["d_oi"] = mdc["d_oi"].where(rs.gap_days(mdc) <= 4)  # no change across a data gap either
RAW = {}  # "opt_<name>" -> wide frame: end-of-day DATA day x name
for part, cols in (
    (iv, ("atm_iv", "skew", "term", "d_atm_iv")),
    (hv, ("iv_hv",)),
    (occ, ("cust_pc", "vol_surge")),
    (mdc, ("oi_pc", "vol_pc", "d_oi", "gex")),
):
    part = part[part["underlying"].isin(NAMES)].assign(date=lambda x: pd.to_datetime(x["date"]))
    for c in cols:
        RAW["opt_" + c] = part.pivot_table(index="date", columns="underlying", values=c).reindex(
            columns=NAMES
        )
print(
    f"raw end-of-day option features re-derived from the store, {lo_day} -> {hi_day}: "
    + ", ".join(f"{k[4:]} {v.notna().any(axis=1).sum()} days" for k, v in RAW.items())
)

r3 = np.flatnonzero((code >= k_day - 1) & (code <= k_day + 1))
daily5 = [k for k in ("gap", "prev_ret") if k in COL]
opt5 = "opt_atm_iv" if "opt_atm_iv" in COL else None
titles = ["intraday · ret_open: return since the open (z), changes every grid bar"]
titles += ["daily · gap and prev_ret (vol units): flat within a session"] if daily5 else []
titles += ["options · opt_atm_iv: flat within a session, steps at the next open"] if opt5 else []
fig = make_subplots(
    rows=len(titles), cols=1, shared_xaxes=True, vertical_spacing=0.09, subplot_titles=titles
)
fig.add_trace(
    go.Scatter(
        x=r3,
        y=A[j, r3, COL["ret_open"]],
        mode="lines",
        line={"shape": "hv", "color": plots.SERIES[0], "width": 1.5},
        name="ret_open (intraday)",
    ),
    row=1,
    col=1,
)
row = 1
if daily5:
    row += 1
    for i, k in enumerate(daily5):
        fig.add_trace(
            go.Scatter(
                x=r3,
                y=A[j, r3, COL[k]],
                mode="lines",
                line={"shape": "hv", "color": plots.SERIES[2 + i], "width": 2},
                name=f"{k} (daily)",
            ),
            row=row,
            col=1,
        )
if opt5:
    row += 1
    ax = "" if row == 1 else str(row)
    fig.add_trace(
        go.Scatter(
            x=r3,
            y=A[j, r3, COL[opt5]],
            mode="lines",
            line={"shape": "hv", "color": plots.SERIES[4], "width": 2},
            name="opt_atm_iv (the model input)",
        ),
        row=row,
        col=1,
    )
    for kk in (k_day - 1, k_day):
        if kk < 0 or kk + 1 >= n_sess or SESS[kk] not in RAW[opt5].index:
            continue
        val = RAW[opt5].loc[SESS[kk], SAMPLE_NAME]
        if not np.isfinite(val):
            continue
        x_eod = int(sf[kk + 1]) - 1  # the session's last grid bar: its end-of-day value
        fig.add_trace(
            go.Scatter(
                x=[x_eod],
                y=[val],
                mode="markers",
                marker={"symbol": "diamond", "size": 11, "color": plots.SERIES[5]},
                name="raw end-of-day IV of that session (options store)",
                showlegend=kk == k_day - 1,
            ),
            row=row,
            col=1,
        )
        fig.add_annotation(
            x=int(sf[kk + 1]),
            y=val,
            ax=x_eod,
            ay=val,
            xref=f"x{ax}",
            yref=f"y{ax}",
            axref=f"x{ax}",
            ayref=f"y{ax}",
            showarrow=True,
            arrowhead=2,
            arrowwidth=1.5,
            arrowcolor=plots.SERIES[5],
            text="",
        )
        fig.add_annotation(
            x=x_eod,
            y=val,
            xref=f"x{ax}",
            yref=f"y{ax}",
            text=f"EOD {SESS[kk]:%m-%d}",
            showarrow=False,
            yshift=14,
            font={"size": 10, "color": plots.SERIES[5]},
        )
for kk in range(max(0, k_day - 1), min(n_sess, k_day + 2)):
    fig.add_vline(x=int(sf[kk]) - 0.5, line={"color": plots._MUTED, "width": 1, "dash": "dot"})
mids = [int(sf[kk]) + n_pos / 2 for kk in range(max(0, k_day - 1), min(n_sess, k_day + 2))]
fig.update_xaxes(
    tickvals=mids,
    ticktext=[f"{SESS[kk]:%a %Y-%m-%d}" for kk in range(max(0, k_day - 1), min(n_sess, k_day + 2))],
    row=len(titles),
    col=1,
    title_text="grid step (Darts' time index: no nights, no weekends); dotted = a session's open",
)
plots._base(
    fig,
    title=f"Fig 5b · {SAMPLE_NAME}: intraday inputs move every bar; daily and option inputs step "
    "at the session boundary",
    height=640,
)
fig.update_annotations(font_size=11)
charts.below(fig, -0.14)
fig.show()

## 6 · Daily and option inputs over time (Fig 6, Table 6)

One panel per slow input and one line per name, with one value per session: the value every grid bar of that
session carries. Gaps are missing data. They enter the model as 0; an option source's `has_*` flag is 0 only when
that whole source is missing, and the daily inputs have no flag (Table 3b). Grey marks the warm-up, and the dotted
line is the first fitted model (`optuna.score_from`).

What to look for:
* **Level differences between names.** A global model sees all five at once, so a name-specific level can work as
  a name identifier rather than as a signal.
* **Few independent values.** A slow input has one value per session and name, repeated on every grid bar. A
  `TRAIN_DAYS`-session window therefore holds about `TRAIN_DAYS` × 5 distinct values of it, not thousands of rows.
  A slowly drifting input (vol20, opt_atm_iv, opt_oi_pc) also spans only a narrow range inside one window.
* **Spikes around earnings** (atm_iv, d_atm_iv, vol_surge, gap), around option expiries, and spikes that are data
  errors.

**Table 6** lists the single most extreme value of each slow input, as a robust z: the distance from that name's
median in MAD units. A robust z in the tens is a genuine shock (check the date against earnings and expiries) or a
bad end-of-day snapshot. Either way the model sees it, because inputs are only clipped at ±20 in their own units. On
the default TUNE run:
* The top row, CSCO's `opt_d_oi` on 2026-01-21, is real. It is the January monthly expiry: the expired contracts
  leave the open interest. `opt_d_oi` dips like this after every monthly expiry (Fig 6).
* ANET's `gap` on 2025-08-06 is its earnings gap. CSCO's `opt_d_atm_iv` on 2025-08-15 is the implied-vol drop after
  its 2025-08-13 report: the drop happened on 08-14, so the model sees it at the next open.
* No junk quote tops the table any more: `options_free.iv_features` drops contracts without a real two-sided quote.
  Before that screen, AAL's skew of 2026-01-15 (a 25-delta put quoted 0.01 bid / 2.20 ask) sat at a robust z of 80.

In [ ]:
SLOW_IN = [k for k in INPUTS if GROUPS[k] in ("daily", "options")]
first_fit = SESS[int(code[pts[0]])]
if not SLOW_IN:
    print("no daily or option inputs with these knobs")
else:
    ncol = 4
    nrow = -(-len(SLOW_IN) // ncol)
    fig = make_subplots(
        rows=nrow,
        cols=ncol,
        subplot_titles=SLOW_IN,
        shared_xaxes=True,
        vertical_spacing=0.25 / nrow,
        horizontal_spacing=0.05,
    )
    for i, k in enumerate(SLOW_IN):
        for jj, s in enumerate(NAMES):
            fig.add_trace(
                go.Scatter(
                    x=DAYS,
                    y=A[jj, sf, COL[k]],
                    mode="lines",
                    line={"color": NAME_COLOR[s], "width": 1.2},
                    name=s,
                    legendgroup=s,
                    showlegend=i == 0,
                    hovertemplate=f"{s} · {k}<br>%{{x|%Y-%m-%d}}: %{{y:.3f}}<extra></extra>",
                ),
                row=i // ncol + 1,
                col=i % ncol + 1,
            )
    fig.add_vrect(
        x0=SESS[0],
        x1=SESS[WARMUP],
        fillcolor=plots._MUTED,
        opacity=0.15,
        line_width=0,
        layer="below",
    )
    fig.add_vline(x=first_fit, line={"color": plots._INK, "dash": "dot", "width": 1})
    fig.update_xaxes(type="date")
    plots._base(
        fig,
        title="Fig 6 · the daily and option inputs per session and name (TUNE)",
        height=230 * nrow + 120,
    )
    fig.update_annotations(font_size=11)
    charts.below(fig, -0.25 / nrow)
    fig.show()
    rows6 = []  # the most extreme value of each slow input: robust z against the name's median
    for k in SLOW_IN:
        vals = pd.DataFrame(A[:, sf, COL[k]].T, index=SESS, columns=NAMES)
        med = vals.median()
        z = ((vals - med) / ((vals - med).abs().median() * 1.4826).replace(0, np.nan)).stack()
        if len(z):
            d, s = z.abs().idxmax()
            rows6.append(
                {
                    "input": k,
                    "name": s,
                    "session": f"{d:%Y-%m-%d}",
                    "value": vals.loc[d, s],
                    "name's median": med[s],
                    "robust z": z[(d, s)],
                }
            )
    t6 = pd.DataFrame(rows6).set_index("input").sort_values("robust z", key=abs, ascending=False)
    print(
        "Table 6 · the most extreme value of each slow input (robust z = distance from the "
        "name's median in MAD units), top 8"
    )
    display(t6.head(8).round(3))

## 7 · How the inputs correlate (Fig 7, Table 7)

This is the rank (Spearman) correlation of every pair of inputs, pooled over the five names and the TUNE grid bars
after the warm-up (the `has_*` flags are left out). Blue means the two move together, red means they move
opposite.

Highly correlated inputs are near-duplicates. That is harmless for gradient boosting, but the importance gets split
between them, so a story built on a single input is unreliable. **Table 7** lists the pairs above |0.5|.

In [ ]:
X7 = pd.DataFrame(A[:, REAL, :].reshape(-1, len(INPUTS)), columns=INPUTS)
X7 = X7[[k for k in INPUTS if GROUPS[k] != "flag" and X7[k].nunique() > 1]]  # flags: data or not
rho = X7.rank().corr()
labels = list(rho.columns)
fig = go.Figure(
    go.Heatmap(
        z=rho.to_numpy(),
        x=labels,
        y=labels,
        colorscale=plots.DIVERGING,
        zmin=-1,
        zmax=1,
        zmid=0,
        colorbar={"title": {"text": "rank corr", "side": "top"}, "thickness": 12, "len": 0.5},
        hovertemplate="%{y} vs %{x}: %{z:.2f}<extra></extra>",
    )
)
fig.update_yaxes(autorange="reversed")
fig.update_xaxes(tickangle=-60)
plots._base(
    fig,
    title="Fig 7 · rank correlation of the inputs (pooled over names, TUNE after the warm-up)",
    height=800,
)
fig.show()
upper = rho.where(np.triu(np.ones(rho.shape, bool), 1)).stack().rename("rank corr")
print("Table 7 · the most correlated pairs (|rank corr| >= 0.5)")
display(upper[upper.abs() >= 0.5].sort_values(key=abs, ascending=False).round(2).to_frame())

## 8 · Timing and leakage checks (Table 8, Table 8b, Table 8c)

A forecast is only worth testing if no input knows the future. Each check re-derives one timing claim from scratch:

* **(a)** the daily, option and flag inputs never change inside a session;
* **(b)** the inputs at a grid bar stay the same when every 1-minute bar AFTER it is replaced by a fresh random
  bar: new open, high, low, close and volume. This is done for several random cuts, plus a cut at a session's first
  and last grid bar (Table 8b). The "after the cut changed" column confirms the replacement did change the later
  inputs, so the check is not empty;
* **(c)** every option value on session `d` equals the raw end-of-day feature of the last stored day strictly
  before `d`, and is missing when that day is more than 5 calendar days old. It is re-derived from the store's raw
  files up to the TUNE end only (§5; `d_atm_iv` only across at most 4 calendar days, never across a data gap), so a
  match also shows that no later day leaks into a TUNE value. It is checked in `panel.features`, whose frames must
  be indexed at UTC midnight of each session, and in the model input itself;
* **(c2)** each options file really is the end of the day it is named after. (c) trusts that label, and if a file
  named `e` held the next day's data, (c) would pass while the model read the future. Each source is tested against
  the stored stock data, on the labelled day and on the next session (Table 8c):
  * MarketData's own underlying price, against the stock's close;
  * DoltHub's spot implied by put-call parity, against the stock's close;
  * DoltHub's vol-history IV change, against the chain's ATM IV change;
  * OCC's volume surge, against the stock's own relative volume.

  The labelled day must match clearly better than the next session;
* **(d)** the usable-target mask `valid` is the same-session rule, re-derived from scratch: a target counts only
  when it is finite and its decision bar, `HORIZON` grid bars earlier, lies in the same session. So no target sums
  returns across two sessions, and each session's first `HORIZON` grid bars never count. This is checked at
  `HORIZON` and at 3, where it can bite;
* **(e)** the volatility scale is constant within a session and uses only previous sessions. It is re-computed from
  scratch, it stays the same when the session's own bars are scrambled, and the next session's scale does move;
* **(f)** half-days: on a 13:00 close the stored "regular" panel keeps 13:00–15:59 (the stored rth flag ignores
  half-day closes). The check shows the core masks those bars:
  * `regular_1m` and `regular` are False on exactly those minutes and grid bars;
  * `left` counts down to 0 on the grid bar closing at 13:00 and is −1 after it;
  * `last_1m` marks 12:59;
  * random bars put in their place change no input, target or scale;
  * a forecast that says "buy" on every grid bar holds nothing after 12:59.

  The half-days are also read off the data, independently of `strategy.HALF_DAYS`: they are the sessions whose
  afternoon minutes are mostly zero-volume placeholders;
* **(g)** `optuna.score_from` is the first session with a fitted model, so a study never scores the warm-up;
* **(h)** the clock inputs (future covariates) match the calendar: time of day, weekday, and the monthly-expiry
  week, Monday to Friday of the week holding the month's third Friday;
* **(i)** options files dated on a market holiday (a weekday inside TUNE without a session) never reach the
  model.

(a) to (e), including (c2), must all pass: a failure there means the model can see the future. (f) to (i) check
data quality and correctness: a failure there is a defect to fix, not a leak.

Why (i) exists: DoltHub writes a file on market holidays that copies the previous session. Used as data, it would
make `opt_d_atm_iv`, the change from the previous stored day, read exactly 0 for all five names on the session after
each such holiday, and the real change of the session before the holiday would never be shown. `research.option_frames`
therefore drops store days inside the panel's span that are not sessions; (i) counts the holiday files in the store
and confirms that no session after a holiday reads that all-zero change.

In [ ]:
checks = []
# (a) slow inputs are constant within every session
SLOW = [COL[k] for k in INPUTS if GROUPS[k] in ("daily", "options", "flag")]
if SLOW:
    xs = A[:, :, SLOW]  # names x grid bars x slow inputs
    opening = xs[:, sf[code], :]  # every grid bar's session-opening value
    moved = ~((xs == opening) | (np.isnan(xs) & np.isnan(opening)))
    bad = sum(
        int(pd.DataFrame(moved[jj]).groupby(code).any().to_numpy().sum())
        for jj in range(len(NAMES))
    )
    detail = (
        f"{n_sess * len(NAMES) * len(SLOW):,} (session, name, input) cells; "
        f"{bad} change inside a session"
    )
    checks.append(
        ("(a) daily, option and flag inputs are constant within every session", bad == 0, detail)
    )
else:
    checks.append(
        (
            "(a) daily, option and flag inputs are constant within every session",
            True,
            "none with these knobs",
        )
    )

# (b) nothing at a grid bar depends on a later 1-minute bar
TOL = ck.TOL  # the tolerance of every "unchanged" comparison
rng = np.random.default_rng(CHECK_SEED)
cuts = sorted(
    {int(b) for b in rng.integers(int(sf[WARMUP]), len(code) - 2, 6)}
    | {int(sf[WARMUP + 10]), int(sf[WARMUP + 20]) - 1}  # a session's first and last grid bar
)
rows8b = []
for i, b in enumerate(cuts):
    k = int(g["end"][b]) + 1  # the first 1-minute bar after grid bar b closed
    after = np.arange(len(panel.close)) >= k
    alt = SM["build_inputs"](ck.scrambled(panel, after, CHECK_SEED + i), **KW)
    later = not all(
        np.allclose(inp["past"][s][b + 1 :], alt["past"][s][b + 1 :], **TOL) for s in NAMES
    )
    rows8b.append(
        {
            "cut after grid bar": b,
            "session": g["date"][b],
            "grid bar closes": rs.clock(g["pos"][b], BAR_MINUTES),
            "inputs, y, scale, target up to the cut unchanged": ck.same_inputs(inp, alt, b + 1),
            "inputs after the cut changed": later,
        }
    )
t8b = pd.DataFrame(rows8b)
ok_b = bool(t8b.iloc[:, -2].all() and t8b.iloc[:, -1].all())
checks.append(
    (
        "(b) the inputs at a grid bar ignore every later 1-minute bar",
        ok_b,
        f"{len(cuts)} cuts (Table 8b): all {len(INPUTS)} inputs, y, scale and target up to the cut "
        "identical; the later inputs did change",
    )
)

# (c) options: the value on session d is the end-of-day value of the last stored day before d
OPT = [k for k in INPUTS if GROUPS[k] == "options"]
if OPT:
    utc = all(str(F[k].index.tz) == "UTC" and F[k].index.equals(SESS_UTC) for k in OPT)
    bad_f = bad_m = stale = differs = 0
    for k in OPT:
        raw = RAW[k]
        have = raw.index[raw.notna().any(axis=1)]  # end-of-day days with data for this input
        frame = F[k].reindex(SESS_UTC)[NAMES].to_numpy(float)  # panel.features (session x name)
        model = A[:, sf, COL[k]].T  # what the model reads on each session's first grid bar
        for i, d in enumerate(SESS):
            e = int(have.searchsorted(d)) - 1  # the last data day strictly before d
            if e < 0 or (d - have[e]).days > 5:
                want = np.full(len(NAMES), np.nan)
                stale += 1
            else:
                want = raw.loc[have[e], NAMES].to_numpy(float)
                if d in raw.index:  # the same day's end-of-day value: what a leak would carry
                    today = raw.loc[d, NAMES].to_numpy(float)
                    differs += int(
                        (np.isfinite(today) & np.isfinite(want) & ~np.isclose(today, want)).sum()
                    )
            bad_f += int(not np.allclose(frame[i], want, rtol=1e-9, atol=1e-12, equal_nan=True))
            bad_m += int(
                not np.allclose(
                    model[i], np.clip(want, -20, 20), rtol=1e-5, atol=1e-6, equal_nan=True
                )
            )
    checks.append(
        (
            "(c) an option value on session d = the end-of-day value of the last day before d",
            utc and bad_f == 0 and bad_m == 0,
            f"panel.features indexed at UTC midnight of each session: {utc}; {len(OPT)} inputs x "
            f"{n_sess} sessions x {len(NAMES)} names: {bad_f} mismatches in panel.features, "
            f"{bad_m} in the model input; {stale} input-sessions without an end-of-day day in the "
            f"last 5 days are missing, as they should be; in {differs:,} cells the same day's own "
            "end-of-day value differed and was NOT used",
        )
    )
else:
    checks.append(
        (
            "(c) option values come from the previous end-of-day",
            True,
            "no option inputs with these knobs",
        )
    )

# (c2) each options file is the end of the day it is named after: (c) trusts that label
reg = g["regular_1m"]  # the real session's minutes: a half-day closes at 13:00
day_close = panel.close[reg].groupby(np.asarray(local.date)[reg]).last()
day_vol = panel.volume[reg].groupby(np.asarray(local.date)[reg]).sum()
CL = day_close.set_axis(pd.to_datetime(day_close.index)).reindex(SESS)[NAMES].to_numpy(float)
DV = day_vol.set_axis(pd.to_datetime(day_vol.index)).reindex(SESS)[NAMES]
REL_VOL = np.log(DV / DV.rolling(20, min_periods=10).mean().shift(1)).to_numpy(float)

rows8c = {}
if any(k in COL for k in ("opt_oi_pc", "opt_vol_pc", "opt_d_oi", "opt_gex")) and len(md_raw):
    m = md_raw.groupby(["date", "underlying"], as_index=False)["underlying_price"].median()
    rows8c["MarketData chain"] = ck.vs_close(
        m["underlying_price"],
        m["date"],
        m["underlying"],
        CL,
        "its underlying price vs the close",
        sessions=SESS,
        symbols=NAMES,
    )
if any(k in COL for k in ("opt_atm_iv", "opt_skew", "opt_term", "opt_d_atm_iv")) and len(dolt_raw):
    ch = dolt_raw[dolt_raw["underlying"].isin(NAMES) & (dolt_raw["bid"] > 0)].assign(
        mid=lambda x: (x["bid"] + x["ask"]) / 2,
        dte=lambda x: (pd.to_datetime(x["expiration"]) - pd.to_datetime(x["date"])).dt.days,
    )
    ch = ch[ch["dte"] >= 5]
    ch = ch[ch["expiration"] == ch.groupby(["date", "underlying"])["expiration"].transform("min")]
    pcp = ch.pivot_table(
        index=["date", "underlying", "strike"], columns="cp", values="mid"
    ).dropna()
    pcp = pcp.assign(diff=(pcp["C"] - pcp["P"]).abs()).reset_index()
    atm = pcp.loc[
        pcp.groupby(["date", "underlying"])["diff"].idxmin()
    ]  # the strike nearest the spot
    rows8c["DoltHub chain"] = ck.vs_close(
        atm["strike"] + atm["C"] - atm["P"],
        atm["date"],
        atm["underlying"],
        CL,
        "spot by put-call parity (call - put + strike) vs the close",
        sessions=SESS,
        symbols=NAMES,
    )
if "opt_iv_hv" in COL and len(hv) and len(iv):
    chain_iv = iv.pivot_table(index="date", columns="underlying", values="atm_iv")
    chain_iv = (
        chain_iv.set_axis(pd.to_datetime(chain_iv.index)).reindex(SESS).reindex(columns=NAMES)
    )
    vh = hv[hv["underlying"].isin(NAMES)].assign(
        d=lambda x: pd.to_datetime(x["date"]), ivc=lambda x: x["iv_current"].astype(float)
    )
    vh = vh.sort_values("d")
    vh["d_iv"] = vh.groupby("underlying")["ivc"].diff()
    rows8c["DoltHub vol history"] = ck.vs_series(
        vh["d_iv"],
        vh["d"],
        vh["underlying"],
        chain_iv.diff().to_numpy(float),
        "daily change of its IV vs the chain's ATM IV change",
        sessions=SESS,
        symbols=NAMES,
    )
if any(k in COL for k in ("opt_cust_pc", "opt_vol_surge")) and len(occ):
    rows8c["OCC volume"] = ck.vs_series(
        occ["vol_surge"],
        occ["date"],
        occ["underlying"],
        REL_VOL,
        "volume surge vs the stock's own volume",
        sessions=SESS,
        symbols=NAMES,
    )
t8c = pd.DataFrame.from_dict(rows8c, orient="index")
t8c.index.name = "source"
checks.append(
    (
        "(c2) each options file holds the end of the day it is named after, not a later day",
        bool(t8c["ok"].all()) if len(t8c) else True,
        "; ".join(
            f"{s}: {r['labelled day']} on the labelled day, {r['next session']} on the next"
            for s, r in rows8c.items()
        )
        or "no option inputs with these knobs",
    )
)

# (d) valid = a finite target whose decision bar (H grid bars earlier) is in the same session
rows8d = []
for hh in sorted({HORIZON, 3}):
    ih = inp if hh == HORIZON else SM["build_inputs"](panel, **{**KW, "horizon": hh})
    cc, vm = ih["g"]["code"], ih["valid"]
    dec_sess = np.r_[np.full(hh, -1), cc[:-hh]]  # the session of each target's decision bar
    rule = np.isfinite(ih["target"]) & (dec_sess == cc)[:, None]  # the rule, from scratch
    p = np.flatnonzero(vm.any(axis=1))
    spans = any(not np.array_equal(cc[p - h], cc[p]) for h in range(1, hh + 1))
    heads = bool(vm[(ih["g"]["sess_first"][:, None] + np.arange(hh)).ravel()].any())
    rows8d.append((hh, bool(np.array_equal(vm, rule)) and not spans and not heads, int(vm.sum())))
checks.append(
    (
        "(d) a target counts only if its decision bar is in the same session",
        all(ok for _, ok, _ in rows8d),
        "; ".join(
            f"horizon {h}: {n:,} usable targets = finite and decided in the same session "
            f"(re-derived), none spans a close, the first {h} grid bar(s) of every session unusable"
            for h, _, n in rows8d
        ),
    )
)

# (e) the scale uses only previous sessions
sc = inp["scale"]
const = bool((pd.DataFrame(sc).groupby(code).nunique(dropna=False) <= 1).all().all())
ret_raw = SM["intraday_features"](panel, g, vol_days=VOL_DAYS)["ret"]  # raw grid-bar log returns
ms = pd.DataFrame(ret_raw**2).groupby(code).mean().to_numpy()  # per-session mean square
need = max(3, VOL_DAYS // 2)
manual = np.full((n_sess, len(NAMES)), np.nan)
for d in range(1, n_sess):
    win = ms[max(0, d - VOL_DAYS) : d]  # the previous VOL_DAYS sessions only
    enough = np.isfinite(win).sum(axis=0) >= need
    manual[d, enough] = np.sqrt(np.nanmean(win[:, enough], axis=0))
match = bool(np.allclose(sc[sf], manual, rtol=1e-6, equal_nan=True))
d0 = WARMUP + 30
rows_d0 = np.flatnonzero(g["codes_1m"] == d0)
f = np.ones(panel.close.shape)
f[rows_d0] = np.exp(
    np.cumsum(
        np.random.default_rng(CHECK_SEED).normal(0.0, 0.003, (len(rows_d0), len(NAMES))), axis=0
    )
)
alt = SM["build_inputs"](
    replace(
        panel, open_=panel.open_ * f, high=panel.high * f, low=panel.low * f, close=panel.close * f
    ),
    **KW,
)
own = bool(np.allclose(sc[code == d0], alt["scale"][code == d0], equal_nan=True))
nxt = not np.allclose(sc[code == d0 + 1], alt["scale"][code == d0 + 1], equal_nan=True)
checks.append(
    (
        "(e) the volatility scale uses only previous sessions",
        const and match and own and nxt,
        f"constant within each session: {const}; equal to a from-scratch RMS of the previous "
        f"{VOL_DAYS} sessions: {match}; scrambling session {SESS[d0]:%Y-%m-%d} leaves its own "
        f"scale unchanged: {own}, and moves the next session's: {nxt}",
    )
)

# (f) half-days: the stored "regular" panel keeps 13:00-15:59 after a 13:00 close; the core masks it
day_1m = np.asarray(local.strftime("%Y-%m-%d"))
mins = np.asarray(local.hour * 60 + local.minute)
on_half = np.isin(day_1m, HALF)  # 1-minute rows of a half-day
late = on_half & (mins >= 13 * 60)  # ... from its 13:00 close on
pm = mins >= 13 * 60  # the calendar read off the data: afternoons of zero-volume placeholders
zero_pm = pd.Series((panel.volume.to_numpy()[pm] == 0).mean(axis=1)).groupby(day_1m[pm]).mean()
found = sorted(zero_pm.index[zero_pm > 0.3])
gday = pd.to_datetime(pd.Series(g["date"])).dt.strftime("%Y-%m-%d").to_numpy()
n_real = np.where(np.isin(gday, HALF), (13 * 60 - 570) // BAR_MINUTES, g["n_pos"])  # by the clock
b_late = g["pos"] >= n_real  # grid bars after a 13:00 close
masks_ok = bool(
    np.array_equal(g["regular_1m"], ~late)
    and np.array_equal(g["regular"], ~b_late)
    and np.array_equal(g["left"], np.where(b_late, -1, n_real - 1 - g["pos"]))
    and np.array_equal(g["last_1m"], mins == np.where(on_half, 13 * 60 - 1, 16 * 60 - 1))
)
unseen = (  # random bars after the 13:00 close change nothing
    ck.same_inputs(inp, SM["build_inputs"](ck.scrambled(panel, late, CHECK_SEED), **KW))
    if late.any()
    else True
)
always = {  # a forecast that says "buy" on every grid bar, traded with no health gate
    "inputs": inp,
    "edge": np.ones(inp["target"].shape),
    "p_up": np.full(inp["target"].shape, np.nan),
}
tf = SM["trade_frames"](panel, always, edge_bp=0.0, health_days=0)
# the holding per bar: an entry while held is trade_frames' re-entry marker, not a new position
state = np.where(tf["entries"].to_numpy(), 1.0, np.where(tf["exits"].to_numpy(), 0.0, np.nan))
held = pd.DataFrame(state).ffill().fillna(0.0).to_numpy()
flat = int((held[late] != 0).sum()) == 0 and int((held[g["last_1m"]] != 0).sum()) == 0
late_bars = int(late.sum())
vol_half = panel.volume.to_numpy()[on_half].sum() if late_bars else np.nan
late_vol = panel.volume.to_numpy()[late].sum() / vol_half if late_bars else 0.0
auction = panel.volume.to_numpy()[late & (mins == 13 * 60)].sum() / vol_half if late_bars else 0.0
zero_late = float((panel.volume.to_numpy()[late] == 0).mean()) if late_bars else 0.0
ok_f = masks_ok and unseen and flat and found == HALF
checks.append(
    (
        "(f) the minutes after a half-day's 13:00 close are masked (data quality, not a leak)",
        ok_f,
        f"{len(HALF)} half-days in TUNE ({', '.join(HALF)}); the data's own afternoons of "
        f"placeholders flag the same sessions: {found == HALF}. They store {late_bars:,} "
        f"one-minute bars from 13:00 on ({zero_late:.0%} zero-volume placeholders; "
        f"{late_vol:.0%} of those days' volume, {auction:.0%} in the 13:00 minute: the closing "
        "auction, which a full day prints at 16:00, outside the panel). Masked: regular_1m and "
        "regular False on exactly "
        f"those minutes and their {int(b_late.sum())} grid bars, left = 0 on the grid bar closing "
        f"13:00 and -1 after, last_1m = 12:59: {masks_ok}; random bars in their place change no "
        f"input, y, scale or target: {unseen}; a buy-every-bar forecast "
        f"({int(tf['entries'].to_numpy().sum()):,} entries) holds nothing after 12:59 and nothing "
        f"at any session's last real bar: {flat}",
    )
)

# (g) the study scores from the first fitted model
checks.append(
    (
        "(g) optuna.score_from = the first session with a fitted model",
        first_fit == pd.Timestamp(strat.optuna["score_from"]),
        f"first retrain point {first_fit:%Y-%m-%d} ({WARMUP} warm-up + {TRAIN_DAYS} training "
        f"sessions); config optuna.score_from {strat.optuna['score_from']}",
    )
)
# (h) the clock inputs (future covariates) match the calendar
FN = inp["future_names"]
fut_g = inp["future"][: len(code)]
dts = pd.to_datetime(pd.Series(g["date"]))
month0 = dts.dt.to_period("M").dt.start_time
third_fri = month0 + pd.to_timedelta((4 - month0.dt.weekday) % 7 + 14, unit="D")
# the ISO week of the month's third Friday: its Monday to Friday, holidays or not
expiry = dts.dt.isocalendar().week.to_numpy(int) == third_fri.dt.isocalendar().week.to_numpy(int)
ok_tod = bool(np.allclose(fut_g[:, FN.index("tod_frac")], g["pos"] / g["n_pos"]))
ok_dow = bool(np.allclose(fut_g[:, FN.index("dow")], dts.dt.weekday.to_numpy() / 4))
marked = fut_g[:, FN.index("opex_week")] > 0.5
ok_opex = bool(np.array_equal(marked, expiry))
op_days = sorted(set(gday[marked]))
op_weeks = sorted({pd.Timestamp(d).strftime("%G-W%V") for d in op_days})
checks.append(
    (
        "(h) the clock inputs (future covariates) match the calendar",
        ok_tod and ok_dow and ok_opex,
        f"time of day: {ok_tod}; weekday: {ok_dow}; opex_week marks {len(op_days)} sessions in "
        f"{len(op_weeks)} weeks ({int(marked.sum()):,} grid bars), exactly the Monday-Friday "
        f"weeks of each month's third Friday: {ok_opex}",
    )
)

# (i) options files dated on a market holiday: a weekday inside TUNE without a session
holidays = pd.bdate_range(SESS[0], SESS[-1]).difference(SESS)
rows8i = {}
for src, folder in STORE_OF.items():
    files = copies = 0
    for h in holidays:
        cur = rs.store_day(folder, f"{h:%Y-%m-%d}", NAMES)
        if len(cur):
            files += 1
            before = SESS[SESS.searchsorted(h) - 1]  # the session before the holiday
            copies += int(cur.equals(rs.store_day(folder, f"{before:%Y-%m-%d}", NAMES)))
    rows8i[src] = (files, copies)
after_h = SESS[SESS.searchsorted(holidays)].tz_localize("UTC")  # the session after each one
d_iv = F["opt_d_atm_iv"].reindex(after_h)[NAMES] if "opt_d_atm_iv" in F else pd.DataFrame()
zero_d = [f"{d:%Y-%m-%d}" for d, r in d_iv.iterrows() if (r == 0).all()]
n_hol_files = sum(n for n, _ in rows8i.values())
hol_txt = ", ".join(f"{h:%Y-%m-%d}" for h in holidays)
checks.append(
    (
        "(i) holiday-dated options files never reach the model (data quality, not a leak)",
        not zero_d,
        f"{len(holidays)} market holidays inside TUNE ({hol_txt}); holiday files in the store: "
        + ", ".join(
            f"{SOURCE_LABEL[s]} {n}"
            + (f" ({c} of {n} exact copies of the session before)" if n else "")
            for s, (n, c) in rows8i.items()
        )
        + f"; research.option_frames drops them. Sessions after a holiday reading opt_d_atm_iv = 0 "
        f"for all {len(NAMES)} names: {', '.join(zero_d) or 'none'}",
    )
)
CHECKS = rs.checks_table(checks)
print("Table 8 · CHECKS")
with pd.option_context("display.max_colwidth", None):
    display(CHECKS)
print("Table 8b · the cuts of check (b)")
display(t8b)
if len(t8c):
    print("Table 8c · check (c2): is each options file the end of the day it is named after?")
    with pd.option_context("display.max_colwidth", 80):
        display(t8c)

## 9 · A first univariate look: TUNE only, and not a model (Fig 9, Table 9)

For each input on its own, this computes the rank correlation (IC) between its value at decision bar `s` and the
realized return of the next `HORIZON` grid bars. It is pooled over the names and the TUNE grid bars after the
warm-up.
* The **t-statistic is clustered by day**. The five names share each day's market, and a daily input repeats on
  every bar of its day, so the bars of one day are not independent.
* The **grey diamonds** repeat the exercise on the **coin-flip market** (`research.sign_surrogate`). There every
  1-minute move is flipped up or down at random: the same volatility, volume and clock, but no direction. The spread
  of the grey t's is what noise looks like.

Four cautions when reading it:
* **Multiple testing.** With 30-odd inputs, one or two clear |t| = 2 (dotted) by luck alone. |t| ≥ 3 (dashed) is
  roughly the bar that survives a Bonferroni-style correction.
* **In-sample.** This uses all of TUNE at once. The walk-forward in `darts_forecast_model.ipynb` is the real test.
* **Period effects.** A flag, or a slowly trending input, can show an IC that only compares two periods.
  `has_oi` is 0 before MarketData's history starts in October 2025 and 1 after, so its IC compares the returns
  before and after that date. It is not a signal.
* **Size.** An IC turns into money at roughly IC × 0.8 × the bar's typical move, against the round-trip cost (both
  printed below; `research.round_trip_bp`, 14 bp at this strategy's $2,000 orders). A "significant" IC can be far
  too small to trade.

In [ ]:
H = HORIZON
FUT = inp["future"]  # the clock, one row per grid bar (+ the bars after the panel)

uni, constant = rs.univariate(inp, g, after_warmup, FUT, H, NAMES)
null, _ = rs.univariate(
    SM["build_inputs"](rs.sign_surrogate(panel, NULL_SEED), **KW), g, after_warmup, FUT, H, NAMES
)
uni["null t"] = null["t"].reindex(uni.index)
uni["group"] = [GROUPS.get(k, "future") for k in uni.index]
order = list(uni["t"].abs().sort_values().index)  # largest |t| at the top

fig = go.Figure()
for grp in dict.fromkeys(uni.loc[order, "group"]):
    sel = [k for k in order if uni.loc[k, "group"] == grp]
    fig.add_trace(
        go.Bar(
            y=sel,
            x=uni.loc[sel, "t"],
            orientation="h",
            name=grp,
            marker_color=GROUP_COLOR[grp],
            customdata=uni.loc[sel, ["ic"]].to_numpy(),
            hovertemplate="<b>%{y}</b><br>t %{x:.2f} · IC %{customdata[0]:.4f}<extra></extra>",
        )
    )
fig.add_trace(
    go.Scatter(
        y=order,
        x=uni.loc[order, "null t"],
        mode="markers",
        marker={"symbol": "diamond", "color": plots._MUTED, "size": 7},
        name="coin-flip market (no direction)",
    )
)
for xv, dash in ((-3, "dash"), (-2, "dot"), (2, "dot"), (3, "dash")):
    fig.add_vline(x=xv, line={"color": plots._INK, "dash": dash, "width": 1})
fig.update_yaxes(categoryorder="array", categoryarray=order)
lim = max(3.5, float(np.nanmax(np.abs(uni[["t", "null t"]].to_numpy(float)))) + 0.5)
fig.update_xaxes(title_text="t-statistic of the rank IC (clustered by day)", range=[-lim, lim])
fig.update_layout(barmode="overlay")
plots._base(
    fig,
    title=f"Fig 9 · each input alone vs the next {H} grid bar(s)' return (TUNE, in-sample)",
    height=22 * len(order) + 170,
)
charts.below(fig, -60 / (22 * len(order) + 170))
fig.show()

cols9 = ["group", "ic", "t", "null t", "n", "days", *NAMES]
print("Table 9 · univariate rank IC per input (per-name columns: that name's own rank IC)")
display(uni.loc[order[::-1], cols9].round(4))
k_best = uni["t"].abs().idxmax()
sigma_bp = float(np.nanmedian(inp["scale"][after_warmup])) * 1e4 * np.sqrt(H)
rt_bp = rs.round_trip_bp(settings, panel)  # the engine's cost at this strategy's order size
print(
    f"{len(uni)} inputs: {int((uni['t'].abs() >= 2).sum())} with |t| >= 2 (luck alone: about "
    f"{0.046 * len(uni):.1f}), {int((uni['t'].abs() >= 3).sum())} with |t| >= 3; on the coin-flip "
    f"market {int((uni['null t'].abs() >= 2).sum())} and {int((uni['null t'].abs() >= 3).sum())}"
)
print(
    f"strongest: {k_best} (IC {uni.loc[k_best, 'ic']:+.4f}, t {uni.loc[k_best, 't']:+.1f}); "
    f"worth about {0.8 * abs(uni.loc[k_best, 'ic']) * sigma_bp:.2f} bp per traded grid bar before "
    f"costs (the typical {H}-bar move is {sigma_bp:.0f} bp), against a {rt_bp:.0f} bp round trip"
)
print(f"constant after the warm-up, so no IC: {', '.join(constant) or 'none'}")

In [ ]:
# One machine-readable line, like every notebook of this repo.
CHECK_ID = CHECKS["check"].str.split(" ").str[0]  # "(a)", ..., "(c2)", ..., "(i)"
leak_ok = bool(CHECKS["ok"][CHECK_ID.isin(["(a)", "(b)", "(c)", "(c2)", "(d)", "(e)"])].all())
rs.scorecard(
    {
        "notebook": "darts_forecast_data",
        "bar_minutes": int(BAR_MINUTES),
        "horizon": int(HORIZON),
        "options": OPTIONS,
        "use_daily": bool(USE_DAILY),
        "use_market": bool(USE_MARKET),
        "tune": [f"{SESS[0]:%Y-%m-%d}", f"{SESS[-1]:%Y-%m-%d}"],
        "sessions": int(n_sess),
        "grid_bars": int(len(code)),
        "inputs": len(INPUTS),
        "flags": [k for k in INPUTS if GROUPS[k] == "flag"],
        "retrain_points": int(len(pts)),
        "first_fit": f"{first_fit:%Y-%m-%d}",
        "scored_forecasts": int(scored.sum()),
        "leakage_checks_pass": leak_ok,
        "checks": {k: bool(v) for k, v in zip(CHECK_ID, CHECKS["ok"], strict=True)},
        "half_days": HALF,
        "half_day_post_close_bars": late_bars,
        "half_day_bars_masked": bool(ok_f),
        "opex_sessions": len(op_days),
        "holiday_option_files": int(n_hol_files),
        "post_holiday_zero_d_atm_iv": zero_d,
        "inputs_abs_t_ge_3": int((uni["t"].abs() >= 3).sum()),
        "null_abs_t_ge_3": int((uni["null t"].abs() >= 3).sum()),
        "constant_inputs": constant,
        "strongest_input": k_best,
        "strongest_ic": float(uni.loc[k_best, "ic"]),
        "strongest_t": float(uni.loc[k_best, "t"]),
        "round_trip_bp": float(rt_bp),
    }
)